In [2]:
import csv
import psycopg2

def map_relationship(value):
    if not value or value.strip() == '' or value.strip() == '0':
        return 'Other'
    v = value.strip().upper()
    if v in ['HERMANO(A)', 'HERMANO', 'MADRE', 'PADRE', 'HIJO(A)', 'NUERA', 'SUEGRO(A)', 'CUÑADO(A)', 'TIO(A)', 'SOBRINO(A)', 'PRIMO(A)']:
        return 'Family'
    if v in ['ESPOSO', 'ESPOSA', 'CONYUGE']:
        return 'Spouse'
    if v in ['AMIGO(A)', 'AMGO(A)']:
        return 'Friend'
    return 'Other'

def parse_null(value):
    if not value or value.strip() == '' or value.strip() == '0':
        return None
    return value.strip()

# Configura tus datos de conexión
DB_HOST = '69.48.206.219'
DB_PORT = '5432'
DB_NAME = 'collection_db'
DB_USER = 'cobranza'
DB_PASS = 'cobranza2025'

def main():
    conn = psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASS,
        host=DB_HOST,
        port=DB_PORT
    )
    cur = conn.cursor()

    with open('../data/debtor_references.csv', newline='', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            name = parse_null(row['name'])
            relationship = map_relationship(row['relationship'])
            phone = parse_null(row['phone'])
            # Si los 3 campos son nulos, no insertar
            if not name and not relationship and not phone:
                continue
            cur.execute("""
                INSERT INTO collection.debtor_references (
                    debtor_id, name, relationship, phone
                ) VALUES (
                    %(debtor_id)s, %(name)s, %(relationship)s, %(phone)s
                )
            """, {
                'debtor_id': int(row['debtor_id']),
                'name': name,
                'relationship': relationship,
                'phone': phone
            })
    conn.commit()
    cur.close()
    conn.close()
    print("Referencias insertadas correctamente.")

if __name__ == '__main__':
    main()

Referencias insertadas correctamente.
